# Padronizando a tabela Mensal

In [0]:
%sql
-- 1. Padronizando a tabela Mensal (SEM a coluna região)
WITH historico AS (
    SELECT
        'Histórico Mensal' AS tipo_pesquisa,
        mes_referencia AS data_referencia,
        produto,
        estado,
        municipio,
        unidade_de_medida,
        numero_postos_pesquisados,
        preco_medio_revenda,
        desvio_padrao_revenda,
        preco_minimo_revenda,
        preco_maximo_revenda,
        margem_media_revenda,
        coef_variacao_revenda,
        preco_medio_distribuicao,
        desvio_padrao_distribuicao,
        preco_minimo_distribuicao,
        preco_maximo_distribuicao,
        coef_variacao_distribuicao
    FROM projeto_combustiveis.silver.historico_mensal_municipios
),

-- 2. Padronizando a tabela Semanal (SEM a coluna região e preenchendo as colunas financeiras ausentes)
semanal AS (
    SELECT
        'Monitor Semanal' AS tipo_pesquisa,
        data_referencia,
        produto,
        estado,
        municipio,
        unidade_de_medida,
        numero_postos_pesquisados,
        preco_medio_revenda,
        desvio_padrao_revenda,
        preco_minimo_revenda,
        preco_maximo_revenda,
        CAST(NULL AS DOUBLE) AS margem_media_revenda,
        coef_variacao_revenda,
        CAST(NULL AS DOUBLE) AS preco_medio_distribuicao,
        CAST(NULL AS DOUBLE) AS desvio_padrao_distribuicao,
        CAST(NULL AS DOUBLE) AS preco_minimo_distribuicao,
        CAST(NULL AS DOUBLE) AS preco_maximo_distribuicao,
        CAST(NULL AS DOUBLE) AS coef_variacao_distribuicao
    FROM projeto_combustiveis.silver.monitor_semanal_municipios
),

-- 3. Empilhando (UNION ALL) os dois históricos
dados_unificados AS (
    SELECT * FROM historico
    UNION ALL
    SELECT * FROM semanal
)

-- 4. Criando a Tabela Fato Final cruzando com as Dimensões
SELECT
    t.id_tempo,
    l.id_localidade,
    p.id_produto,
    u.tipo_pesquisa,
    u.unidade_de_medida,
    u.numero_postos_pesquisados,
    u.preco_medio_revenda,
    u.desvio_padrao_revenda,
    u.preco_minimo_revenda,
    u.preco_maximo_revenda,
    u.margem_media_revenda,
    u.coef_variacao_revenda,
    u.preco_medio_distribuicao,
    u.desvio_padrao_distribuicao,
    u.preco_minimo_distribuicao,
    u.preco_maximo_distribuicao,
    u.coef_variacao_distribuicao
FROM dados_unificados u

INNER JOIN projeto_combustiveis.gold.dim_tempo t 
    ON u.data_referencia = t.data_referencia

INNER JOIN projeto_combustiveis.gold.dim_produto p 
    ON u.produto = p.produto

INNER JOIN projeto_combustiveis.gold.dim_localidade l 
    ON u.estado = l.estado 
    AND u.municipio = l.municipio

In [0]:
df_fato_consolidada = _sqldf

df_fato_consolidada.write.mode("overwrite").saveAsTable("projeto_combustiveis.gold.fact_precos_combustiveis")
display(df_fato_consolidada)